<img src="https://github.com/nicholasmetherall/digital-earth-pacific-macblue-activities/blob/main/attachments/images/DE_Pacific_banner.JPG?raw=true" width="900"/>

Figure 1.1.a. Jupyter environment + Python notebooks

# Digital Earth Pacific Notebook 1B prepare GeoMAD and do vegetation analyses

The objective of this notebook is to prepare a geomad postcard for your AOI (masking, scaling and loading additional band ratios and spectral indices) in order to do vegetation line analysis.

## Step 1.1: Configure the environment

In [1]:
import os
from datetime import datetime
from shapely.geometry import Polygon
from shapely import box
from pyproj import CRS 
import folium
from odc.geo import Geometry
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio as rio
import xarray as xr
import rioxarray
from ipyleaflet import basemaps
from numpy.lib.stride_tricks import sliding_window_view
import pystac_client
from dask.distributed import Client as DaskClient
from odc.stac import load, configure_s3_access
import planetary_computer
# from odc.stac import load
from pystac.client import Client
from skimage.feature import graycomatrix, graycoprops
from utils import load_data, load_s1_dem, scale, calculate_band_indices, apply_mask, mask_land, all_masks, do_prediction

In [2]:
# Reload scripts and imports
%load_ext autoreload
%autoreload 2

<font color='red'>1.1. Revise the site of training data...


In [16]:
site = "Mejrirok"
filename = f"inputs/{site}-site.geojson"

## Step 1.2: Configure STAC access and search parameters

Do you remember what a stac catalog is...? You can insert this one : https://stac.digitalearthpacific.org


In [17]:
catalog = "https://stac.digitalearthpacific.org"
client = Client.open(catalog)

<font color='red'>1.2. Revise the site name

In [18]:
area_of_interest = "Mejrirok"

# 1. Read the file
aoi = gpd.read_file(f"{filename}")

# 2. Get the bounding box directly from the GeoDataFrame
# This returns exactly what STAC needs: (minx, miny, maxx, maxy)
bbox = aoi.total_bounds 

<font color='red'>1.3. Revise the year to `2025` and include the stac catalog collections `dep_s2_geomad`

In [19]:
# 3. Proceed with your search
year = '2025'
items = list(
    client.search(
        collections=["dep_s2_geomad"],
        bbox=bbox,
        datetime=year
    ).items()
)

#### How many images were found?

In [20]:
print(f"Found {len(items)} items in for {datetime}")

Found 1 items in for <class 'datetime.datetime'>


<font color='red'>1.4. Define the spectral bands you are interested in as:
> `measurements = ["nir", "red", "blue", "green", "emad", "smad", "bcmad", "green", "nir08", "nir09", "swir16", "swir22", "coastal", "rededge1", "rededge2", "rededge3"]`

In [21]:
measurements = ["nir", "red", "blue", "green", "emad", "smad", "bcmad", "green", "nir08", "nir09", "swir16", "swir22", "coastal", "rededge1", "rededge2", "rededge3"]

<font color='red'>1.5. Load the data

In [22]:
data = load_data(
    items,
    measurements,
    bbox,
)

In [16]:
# dask_client = DaskClient(n_workers=1, threads_per_worker=16, memory_limit='32GB')
# configure_s3_access(cloud_defaults=True, requester_pays=True)

<font color='red'>1.6. Scale the variable 'data'
>scaled = scale(`insertvariable`)

In [23]:
scaled = scale(data)

In [24]:
scaled = scaled.squeeze()

<font color='red'>1.7. Explore the new 'scaled' variable 
>`insertvariable`.odc.explore(vmin=0, vmax=0.3, bands=["red", "green", "blue"], crs="EPSG:3832", name=site)

In [25]:
scaled.odc.explore(vmin=0, vmax=0.3, bands=["red", "green", "blue"], crs="EPSG:3832", name=site)

/srv/conda/envs/notebook/lib/python3.11/site-packages/rasterio/warp.py:387: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dest = _reproject(


<font color='red'>1.8. Calculate the spectral bands for this 'scaled' variable 
>`variable` = calculate_band_indices(`variable`)  

In [26]:
scaled = calculate_band_indices(scaled)

In [27]:
# # see results by inserting variable and running code below
scaled

<xarray.Dataset> Size: 1MB
Dimensions:        (y: 108, x: 51)
Coordinates:
  * y              (y) float64 864B 6.502e+05 6.502e+05 ... 6.492e+05 6.492e+05
  * x              (x) float64 408B 2.173e+06 2.173e+06 ... 2.174e+06 2.174e+06
    spatial_ref    int32 4B 3832
    time           datetime64[ns] 8B 2025-01-01
Data variables: (12/32)
    nir            (y, x) float64 44kB dask.array<chunksize=(108, 51), meta=np.ndarray>
    red            (y, x) float64 44kB dask.array<chunksize=(108, 51), meta=np.ndarray>
    blue           (y, x) float64 44kB dask.array<chunksize=(108, 51), meta=np.ndarray>
    green          (y, x) float64 44kB dask.array<chunksize=(108, 51), meta=np.ndarray>
    emad           (y, x) float32 22kB dask.array<chunksize=(108, 51), meta=np.ndarray>
    smad           (y, x) float32 22kB dask.array<chunksize=(108, 51), meta=np.ndarray>
    ...             ...
    ndci           (y, x) float64 44kB dask.array<chunksize=(108, 51), meta=np.ndarray>
    nbi            (y, x) float64 44kB dask.array<chunksize=(108, 51), meta=np.ndarray>
    ndmi           (y, x) float64 44kB dask.array<chunksize=(108, 51), meta=np.ndarray>
    bsi            (y, x) float64 44kB dask.array<chunksize=(108, 51), meta=np.ndarray>
    awei           (y, x) float64 44kB dask.array<chunksize=(108, 51), meta=np.ndarray>
    tc_wetness     (y, x) float64 44kB dask.array<chunksize=(108, 51), meta=np.ndarray>

### How many data variables are there in total?

### Explore NDVI

Add some markdown to explain NDVI

Input formula to define the index:  
`data["ndvi"] = (data["nir"] - data["red"]) / (data["nir"] + data["red"])`


In [28]:
data["ndvi"] = (data["nir"] - data["red"]) / (data["nir"] + data["red"])

In [39]:
ndvi_mask = (data.ndvi).squeeze() > 0.9

In [42]:
ndvi_mask.odc.explore()

### Explore EVI

Add some markdown to explain EVI

Input formula to define the index:   
`data["evi"] = (2.5 * data["nir"] - data["red"]) / (data["nir"] + (6 * data["red"]) - (7.5 * data["blue"]) + 1)`

### Explore SAVI

Add some markdown to explain SAVI

Input formula to define the index:  
`data["savi"] = (data["nir"] - data["red"]) / (data["nir"] + data["red"])`

<font color='red'>1.9. Mask out the 'scaled' variable including the ocean 
>`scaled`, mask = all_masks(`scaled`, return_mask = True)


<font color='red'>Explore the extent of this 'mask' variable  
>`insertvariable`.odc.explore(vmin=0, vmax=0.3, bands=["red", "green", "blue"], crs="EPSG:3832", name=site)

<font color='red'>2.5. Export your final `raster` variable to a geojson file.
>`variable`.to_file(f"output/{site}-vector.geo", index=False)

<font color='red'>2.5. Export your final `vector` variable to a geojson file.
>`variable`.to_file(f"output/{site}-vector.geo", index=False)